# WooCommerce Order Poller (Phase 2)

This notebook is a **separate, independent process** from the synthetic data generator. Run it in its own Jupyter kernel, at the same time as (or after) the generator.

**What it does:**
1. Remembers the timestamp it last checked, in a small state file.
2. Every few seconds, asks WooCommerce: "any orders created or updated since then?"
3. Saves each one as its own `.json` file in a landing folder on your machine.
4. Updates its "last checked" timestamp and repeats.

That landing folder is what Spark Structured Streaming will read from in Phase 3 — every new file that lands is treated as a new streaming event.

**Note:** because this polls on `modified_after` (not just `after`), it captures both brand-new orders *and* status changes (e.g. an order going from `pending` -> `processing` -> `completed` will land as a fresh file each time), so you get the full order lifecycle, not just creation events.


In [1]:
!pip install woocommerce requests -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os

from woocommerce import API
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ---- Config: same store as the generator. Fill these in, or set as environment variables ----
WC_URL = os.environ.get("WC_URL", "")
WC_CONSUMER_KEY = os.environ.get("WC_CONSUMER_KEY", "")
WC_CONSUMER_SECRET = os.environ.get("WC_CONSUMER_SECRET", "")

if not all([WC_URL, WC_CONSUMER_KEY, WC_CONSUMER_SECRET]):
    raise EnvironmentError(
        "Set WC_URL, WC_CONSUMER_KEY, WC_CONSUMER_SECRET (env vars or directly above)."
    )

wcapi = API(
    url=WC_URL,
    consumer_key=WC_CONSUMER_KEY,
    consumer_secret=WC_CONSUMER_SECRET,
    version="wc/v3",
    timeout=20,
    verify_ssl=False,  # needed for LocalWP's self-signed local SSL certificate
)

In [4]:
print("Key:", repr(WC_CONSUMER_KEY))
print("Secret:", repr(WC_CONSUMER_SECRET))
print("URL:", repr(WC_URL))

Key: 'ck_854909d96800af1a046423e0a29675d148d3be7b'
Secret: 'cs_470f6473f81f7584243b9923b83df56744f97409'
URL: 'https://retail-analytics.local'


## Connectivity check
Same as the generator notebook — confirms this notebook's config works before doing anything else.

In [5]:
resp = wcapi.get("orders", params={"per_page": 1})
print("Status code:", resp.status_code)
if resp.status_code == 200:
    print("Auth OK.")
else:
    print("Problem:", resp.text[:300])

Status code: 200
Auth OK.


## Landing folder and state file
`LANDING_DIR` is where every order snapshot gets saved as a `.json` file — point Spark at this folder in Phase 3.

`STATE_FILE` remembers the last timestamp this poller checked, so stopping and restarting it doesn't reprocess everything from scratch.

In [6]:
import json
from pathlib import Path

LANDING_DIR = Path("woo_orders_landing")
LANDING_DIR.mkdir(exist_ok=True)

STATE_FILE = Path("poller_state.json")

def load_last_seen():
    if STATE_FILE.exists():
        with open(STATE_FILE, "r") as f:
            state = json.load(f)
        return state.get("last_modified_seen", "2000-01-01T00:00:00")
    return "2000-01-01T00:00:00"

def save_last_seen(timestamp):
    with open(STATE_FILE, "w") as f:
        json.dump({"last_modified_seen": timestamp}, f)

print("Landing folder:", LANDING_DIR.resolve())
print("Resuming from:", load_last_seen())

Landing folder: C:\Users\chait\Documents\spark_projects\woo_orders_landing
Resuming from: 2026-07-28T07:25:27


## Fetch and land functions

In [7]:
def fetch_new_orders(last_seen, per_page=100):
    """Fetch orders created or modified after last_seen. Returns a list sorted oldest-first."""
    resp = wcapi.get("orders", params={
        "modified_after": last_seen,
        "dates_are_gmt": True,
        "per_page": per_page,
        "orderby": "modified",
        "order": "asc",
    })
    if resp.status_code != 200:
        print("Poll failed:", resp.status_code, resp.text[:200])
        return []
    orders = resp.json()
    orders.sort(key=lambda o: o["date_modified_gmt"])
    return orders


def write_order_file(order):
    """Land one order snapshot as a JSON file. Filename encodes id + modified time, so the
    same order re-landing later (e.g. after a status change) creates a new distinct file."""
    safe_ts = order["date_modified_gmt"].replace(":", "-")
    order_id = order["id"]
    filename = f"order_{order_id}_{safe_ts}.json"
    filepath = LANDING_DIR / filename
    with open(filepath, "w") as f:
        json.dump(order, f)
    return filepath

## Main poll loop
**Notebook note:** `n_polls` defaults to a finite number so this cell doesn't block forever. Bump it up, or set it to `None` and use the kernel's Interrupt/Stop button when you want to stop an indefinite run.

In [8]:
import time
import random
from datetime import datetime

def run_poller(interval_range=(5, 15), n_polls=20):
    last_seen = load_last_seen()
    print(f"[{datetime.now()}] Starting poller. Resuming from {last_seen}")

    polls_done = 0
    total_landed = 0
    try:
        while n_polls is None or polls_done < n_polls:
            orders = fetch_new_orders(last_seen)
            for order in orders:
                path = write_order_file(order)
                total_landed += 1
                print(f"[landed] order #{order['id']} (status={order['status']}) -> {path.name}")

            if orders:
                last_seen = orders[-1]["date_modified_gmt"]
                save_last_seen(last_seen)

            polls_done += 1
            time.sleep(random.uniform(*interval_range))
    except KeyboardInterrupt:
        pass

    print(f"[{datetime.now()}] Stopped. Landed {total_landed} order snapshots across {polls_done} polls.")

In [ ]:
run_poller(n_polls=None)

[2026-07-28 13:06:27.363922] Starting poller. Resuming from 2026-07-28T07:25:27
[landed] order #931 (status=pending) -> order_931_2026-07-28T07-36-52.json
[landed] order #932 (status=processing) -> order_932_2026-07-28T07-37-23.json
[landed] order #933 (status=completed) -> order_933_2026-07-28T07-37-41.json
[landed] order #934 (status=completed) -> order_934_2026-07-28T07-37-57.json
[landed] order #935 (status=pending) -> order_935_2026-07-28T07-38-09.json
[landed] order #936 (status=pending) -> order_936_2026-07-28T07-38-27.json
[landed] order #937 (status=pending) -> order_937_2026-07-28T07-38-35.json
[landed] order #935 (status=completed) -> order_935_2026-07-28T07-38-37.json
[landed] order #938 (status=processing) -> order_938_2026-07-28T07-39-00.json
[landed] order #939 (status=processing) -> order_939_2026-07-28T07-39-17.json
[landed] order #940 (status=processing) -> order_940_2026-07-28T07-39-37.json
[landed] order #941 (status=completed) -> order_941_2026-07-28T07-39-57.json


## Peek at what landed
Run this any time — during or after polling — to see the files that have been written so far.

In [ ]:
landed_files = sorted(LANDING_DIR.glob("*.json"))
print(f"{len(landed_files)} files in landing folder")
for f in landed_files[-10:]:
    print(f.name)

25 files in landing folder
order_90_2026-07-26T14-44-44.json
order_91_2026-07-26T14-44-41.json
order_92_2026-07-26T14-44-55.json
order_93_2026-07-26T14-45-10.json
order_94_2026-07-26T14-45-20.json
order_95_2026-07-26T14-47-10.json
order_96_2026-07-26T14-46-00.json
order_97_2026-07-26T14-46-15.json
order_98_2026-07-26T14-46-27.json
order_99_2026-07-26T14-46-39.json
